# Milestone 5 Lab, notebook 4 of 7: A workflow that can wait

## Where we are

| Notebook | What you do |
|---|---|
| 1. Files arrive, by hand | Load the files the naive way, and watch three things go wrong |
| 2. A loader that is safe to repeat | Fingerprint each file, check its columns, and choose a rule for every file |
| 3. AI reads a changed file | Ask a model what a renamed column means, and score its suggestions |
| **4. A workflow that can wait** (you are here) | Build the loader as a graph that pauses for your approval |
| 5. Crash and carry on | Restart the kernel on purpose, and find the file still waiting |
| 6. A swarm of file workers | Load fifty files side by side, and score the run for quality, cost, and time |
| 7. The same question, from loaded files | Approve the waiting files, run the pipeline, and ask the leader's question |

Model requests in this notebook: **about 2**. A full run of this lab costs about 3 US cents.

## What this notebook is for

A person must approve each mapping. That person might be in a meeting, or asleep. So the loader must be able to stop in the middle of a file, wait, and carry on later.

Until now every agent we built ran from start to finish in one go. In this notebook you build a workflow as a **graph**: separate steps joined by arrows, with its progress saved after every step. Then you make it pause for your approval.

The library is **LangGraph**. You have been using it since milestone 1 without knowing.

> This notebook needs `OPENROUTER_API_KEY` in your `.env` file.

In [ ]:
import sqlite3
from typing import TypedDict

import pandas as pd
from lab_helpers import LAB, LANDING, reset_lab, show_doc

reset_lab()

## Step 1: The three ideas

In [ ]:
show_doc("HLD.md", "7. A workflow that can wait: three ideas")

## Step 2: The agent you already know is a graph

Here is the analyst from milestone 1, made with LangChain's `create_agent`. Ask what type of thing it is.

In [ ]:
from langchain.agents import create_agent

from retail_profit_stack.agent import SYSTEM_PROMPT, make_model, make_tools

model = make_model()
analyst = create_agent(model=model, tools=make_tools([]), system_prompt=SYSTEM_PROMPT)

type(analyst)

A `CompiledStateGraph`: a LangGraph graph. LangGraph can describe a graph as text, in a format for diagrams called Mermaid.

In [ ]:
print(analyst.get_graph().draw_mermaid())

Read the lines with arrows. There are two steps, `model` and `tools`. The graph goes back and forth between them until the model stops asking for tools. That loop is the agent loop you built by hand in milestone 1.

Now we build a graph of our own, for one file, with five steps.

## Step 3: The state

A graph has three parts. The first is the **state**: the notes the workflow keeps for one file. Every step can read them, and returns the parts it changes.

In [ ]:
class FileState(TypedDict):
    """What the workflow keeps for one file."""

    file_name: str
    feed: str
    missing: list
    unknown: list
    outcome: str      # load, needs_mapping, rejected, loaded
    mapping: dict     # file column -> expected column
    suggestion: dict  # what the model suggested
    approved_by: str
    rows_loaded: int

## Step 4: The nodes

The second part is the **nodes**. A node is a plain Python function. It takes the state and returns a dictionary with the fields it changes.

The first node inspects the file. It uses `inspect_file` from notebook 2.

In [ ]:
from retail_profit_stack.ingest.feeds import expected_columns, feed_of
from retail_profit_stack.ingest.rules import inspect_file


def inspect(state: FileState):
    """Compare the file's columns with the contract, and choose the rule."""
    feed = feed_of(state["file_name"])
    found = inspect_file(LANDING / state["file_name"], expected_columns(feed), loaded={}, approved=[])
    print("inspect:", state["file_name"], "->", found.outcome)
    return {"feed": feed, "missing": found.missing, "unknown": found.unknown, "outcome": found.outcome}

The second node asks the model for a mapping, with `suggest_mapping` from notebook 3.

In [ ]:
from retail_profit_stack.ingest.mapping import suggest_mapping


def suggest(state: FileState):
    """Ask the model what the unknown columns mean."""
    file = LANDING / state["file_name"]
    suggestion, cost = suggest_mapping(model, state["feed"], file, state["missing"], state["unknown"], LAB)
    print("suggest:", suggestion.as_mapping(), "confidence", suggestion.confidence, "cost", round(cost, 5))
    return {"suggestion": suggestion.model_dump(), "mapping": suggestion.as_mapping()}

The third node is the new idea. `interrupt()` **pauses the graph**. LangGraph saves the state, together with the value we pass to `interrupt()`, and the graph stops there.

Later, when someone resumes the graph with an answer, `interrupt()` returns that answer and the node carries on.

In [ ]:
from langgraph.types import interrupt


def approval(state: FileState):
    """Wait for a person to approve or refuse the mapping."""
    print("approval:", state["file_name"], "needs a person to approve", state["mapping"])
    answer = interrupt({"file_name": state["file_name"], "mapping": state["mapping"]})
    if answer["approved"]:
        return {"outcome": "load", "approved_by": answer["person"]}
    return {"outcome": "rejected", "approved_by": answer["person"]}

The fourth node loads the file, with `read_file` from notebook 3. Here it only counts the rows. The production version also writes them to disk and to the load record.

In [ ]:
from retail_profit_stack.ingest.mapping import read_file


def load(state: FileState):
    """Apply the mapping, check the values, and load the rows."""
    table = read_file(LANDING / state["file_name"], state["feed"], state.get("mapping"))
    print("load:", table.num_rows, "rows")
    return {"outcome": "loaded", "rows_loaded": table.num_rows}

## Step 5: The edges

The third part is the **edges**: which node runs next. A plain edge always goes to the same node. A **conditional edge** is a function that looks at the state and returns the name of the next node.

In [ ]:
from langgraph.graph import END


def after_inspect(state: FileState):
    """Load a file that is as expected. Ask about one that needs a mapping. Stop otherwise."""
    if state["outcome"] == "load":
        return "load"
    if state["outcome"] == "needs_mapping":
        return "suggest"
    return END


def after_approval(state: FileState):
    """Load an approved file. Stop on a refused one."""
    if state["outcome"] == "load":
        return "load"
    return END

## Step 6: Somewhere to save the state

The object that saves the state is called a **checkpointer**. Ours writes to the lab's SQLite file, the same file that holds the load record. `setup()` creates its tables the first time and does nothing after that.

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

connection = sqlite3.connect(LAB.state_file, check_same_thread=False)
checkpointer = SqliteSaver(connection)
checkpointer.setup()

## Step 7: Build the graph

Nodes and edges go into a `StateGraph`. `compile` turns it into something we can run, with the checkpointer attached.

In [ ]:
from langgraph.graph import START, StateGraph

builder = StateGraph(FileState)
builder.add_node("inspect", inspect)
builder.add_node("suggest", suggest)
builder.add_node("approval", approval)
builder.add_node("load", load)

builder.add_edge(START, "inspect")
builder.add_conditional_edges("inspect", after_inspect, ["load", "suggest", END])
builder.add_edge("suggest", "approval")
builder.add_conditional_edges("approval", after_approval, ["load", END])
builder.add_edge("load", END)

graph = builder.compile(checkpointer=checkpointer)

In [ ]:
print(graph.get_graph().draw_mermaid())

Follow the arrows from `__start__`. Dotted arrows (`-.->`) are conditional edges. There are two ways through: straight from `inspect` to `load`, or round by `suggest` and `approval`.

## Step 8: A file that is as expected

Each file runs in its own **thread**: one workflow's saved history. We use the file's name as the thread's id, so we can always find a file's workflow again.

In [ ]:
config = {"configurable": {"thread_id": "orders_2026-09-07.csv"}}

result = graph.invoke({"file_name": "orders_2026-09-07.csv"}, config)

print(result["outcome"], result["rows_loaded"])

Two nodes ran, and the graph finished. No model was called and nobody was asked.

## Step 9: A file that must wait

Now a file with renamed columns.

In [ ]:
config = {"configurable": {"thread_id": "order_items_2026-10-12.csv"}}

result = graph.invoke({"file_name": "order_items_2026-10-12.csv"}, config)

result["__interrupt__"]

The graph **stopped**. `invoke` returned, and the result carries `__interrupt__`: the value we passed to `interrupt()`. That is the question waiting for a person.

Nothing is running now. The state is in the SQLite file. `get_state` reads it back, and `next` says which node runs when the thread carries on.

In [ ]:
saved = graph.get_state(config)

print("Waiting at:", saved.next)
print("Rows loaded so far:", saved.values.get("rows_loaded"))
print("The suggestion:", saved.values["mapping"])

## Step 10: You approve

`Command(resume=...)` carries on the thread, and hands your answer to the `interrupt()` that is waiting. Put your own name in.

In [ ]:
from langgraph.types import Command

answer = {"approved": True, "person": "your name"}

result = graph.invoke(Command(resume=answer), config)

print(result["outcome"], result["rows_loaded"], "rows, approved by", result["approved_by"])

Look at what was printed. The `approval` line appeared **a second time**. On resume, LangGraph runs the interrupted node again from its top, and this time `interrupt()` returns your answer.

So any code before `interrupt()` runs twice. A print is harmless. Loading rows would not be. That is why loading has a node of its own, after the approval.

## Step 11: The production workflow

Production's `build_file_workflow` has the same shape, with one more node, `finish`, that writes the load record. Its `load` node writes the rows to disk as a **piece**, and claims the fingerprint first.

In [ ]:
from retail_profit_stack.ingest.workflow import build_file_workflow

workflow = build_file_workflow(LAB, model)

print(workflow.get_graph().draw_mermaid())

`run_file` runs it for one file and returns a short result. Try the three kinds of file.

In [ ]:
from retail_profit_stack.ingest.workflow import run_file

for file_name in ["refunds_2026-09-21.csv", "refunds_2026-10-12.csv", "payment_fees_2026-10-19.csv"]:
    result = run_file(workflow, file_name)
    print(f"{result['outcome']:9} {file_name:30} {result['reason']}")

One loaded, one was rejected, and one is waiting. `waiting_files` lists what is waiting, and `answer` gives your decision.

In [ ]:
from retail_profit_stack.ingest.workflow import answer, waiting_files

for question in waiting_files(workflow, LAB):
    print(question["file_name"], "suggests", question["suggested_mapping"])

In [ ]:
answer(workflow, "payment_fees_2026-10-19.csv", approved=True, person="your name")

An approved mapping is **saved**, so you are asked once. The next file from the same feed, with the same change, loads without a pause.

In [ ]:
print(LAB.mappings_file.read_text(encoding="utf-8"))

result = run_file(workflow, "payment_fees_2026-10-26.csv")
print(result["outcome"], "-", result["reason"])

In [ ]:
from retail_profit_stack.ingest import ledger

pd.DataFrame(ledger.files_seen(LAB.state_file))[["file_name", "outcome", "rule", "rows_loaded", "approved_by"]]

## What we now know

- A workflow can be drawn as a **graph**: a **state**, **nodes** that change it, and **edges** that choose the next node.
- `create_agent` has been building a LangGraph graph all along.
- A **checkpointer** saves the state after every node. Here it writes to a SQLite file.
- `interrupt()` **pauses** a graph and hands a question out. `Command(resume=...)` carries on with the answer.
- On resume, the paused node runs again from the top. So anything that must not happen twice goes in a later node.
- An approved mapping is saved and reused. A person is asked once.

**Next:** notebook 5. The state is on disk. So what happens if the program dies while a file is waiting?